# Advanced: Scoring single mutations in antibody-antigen complexes with evedesign

> This example demonstrates advanced functionality for dealing with 3D structure information. Also see our simpler basic example that automatically adds structures to the molecular system with FoldSeek.

In this notebook we will show how both ESM-2 and ProteinMPNN can be used through `evedesign` to perform sequence- and structure-based single mutation scanning, respectively. This example is based on the second case study of the `evedesign` manuscript, where we replicated the single-mutation scoring workflow from [Hie *et al.*](https://www.nature.com/articles/s41587-023-01763-2) which aimed to discover antibody variants with increased affinities to particular antigens.

Reference: Hie, B.L., Shanker, V.R., Xu, D. et al. Efficient evolution of human antibodies from general protein language models. Nat Biotechnol 42, 275–283 (2024). https://doi.org/10.1038/s41587-023-01763-2

In [1]:
import torch
import pandas as pd

from evedesign.structure import StructureFile
from evedesign.system import System, Protein, SystemInstance, EntityInstance
from evedesign.model import system_subset_model
from evedesign.models.mpnn import LigandMPNN
from evedesign.models.esm2 import ESM2
from evedesign.types import DeviceType
from evedesign.tools.structure_mapping import map_structure_chain_to_entity_pairwise

DEVICE: DeviceType = "cuda" if torch.cuda.is_available() else "cpu"


/Users/thomashopf/mambaforge/envs/modal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Defining the complex system

First, let's define our target sequences. These were be obtained from the supplementary files from Hie *et al.* and from public databases. In this case, our example will be the S309 antibody in complex with the SARS-CoV-2 spike protein.

In [2]:
ag_ab_seqs = pd.read_csv("antibody_antigen_sequences.csv")

# Target PDB ID and chain IDs; note these are the canonical PDB chain IDs, *not* author chain IDs
target_pdb_id = "6WPS"
vh_chain_id, vl_chain_id, ag_chain_id = "B", "C", "A"

vh_seq = ag_ab_seqs.query("name == 'S309' and L_H == 'H'").iloc[0].sequence
vl_seq = ag_ab_seqs.query("name == 'S309' and L_H == 'L'").iloc[0].sequence
antigen_seq = ag_ab_seqs.query("name == 'CoV-2 WT S6P'").iloc[0].sequence

In [3]:
# PDB structure to use for ProteinMPNN
target_pdb_model = StructureFile.from_id(target_pdb_id).get_model(
    use_author_fields=False
)

vh_chain = target_pdb_model.get_chain(vh_chain_id)
vl_chain = target_pdb_model.get_chain(vl_chain_id)
antigen_chain = target_pdb_model.get_chain(ag_chain_id)

We can now define our complex system. We will attach 3D structures for each chain in a second step as we will need to remap their position indices first.

In [4]:
complex_system = System([
    Protein(
        id="vh", rep=vh_seq, first_index=1, structures={},
    ),
    Protein(
        id="vl", rep=vl_seq, first_index=1, structures={},
    ),
    Protein(
        id="ag", rep=antigen_seq, first_index=1, structures={},
    ),
])

Note that we use the same key `target_pdb_id` for each of the `structures` attributes, which indicates that the coordinates belong together in the same multi-entity complex structure.

In [5]:
for entity_idx, entity_chain in enumerate(
    [vh_chain, vl_chain, antigen_chain]
):
    complex_system[entity_idx].structures[target_pdb_id] = map_structure_chain_to_entity_pairwise(
        complex_system[entity_idx], entity_chain, local=True
    )

    print(
        f"number of residues mapped for entity {entity_idx}:",
        len(complex_system[entity_idx].structures[target_pdb_id].res_df())
    )


number of residues mapped for entity 0: 123
number of residues mapped for entity 1: 102
number of residues mapped for entity 2: 955


Inspect the mapped VH structure chain:

In [6]:
complex_system[0].structures[target_pdb_id].res_df()

,res_id,res_name,ins_code,chain_id,sse,atom_df_start_idx,res_name_oneletter
0,2,VAL,,B,C,0,V
1,3,GLN,,B,E,7,Q
2,4,LEU,,B,E,16,L
3,5,VAL,,B,E,24,V
4,6,GLN,,B,E,31,Q
...,...,...,...,...,...,...,...
118,120,GLY,,B,C,904,G
119,121,THR,,B,C,908,T
120,122,LEU,,B,C,915,L
121,123,VAL,,B,C,923,V


## 2. Build the models

The ProteinMPNN model can be built as follows (parameters will be automatically downloaded):

In [7]:
mpnn_model = LigandMPNN(
    model_name="solublempnn_v_48_002",
    device=DEVICE,
).build(
    complex_system
)

2026-06-19 12:31:40.630 | INFO     | evedesign.models.mpnn:download_checkpoint:99 - Using cached checkpoint at /Users/thomashopf/.cache/mpnn/solublempnn_v_48_002.pt


We can also build a protein language model (ESM-2) that operates on a single entity of the complex system with `system_subset_model`. This helper automatically deals with mapping back and forth to subsets of the full system where desired.

In [8]:
esm = system_subset_model(ESM2)(
    model_name="esm2_t33_650M_UR50D",
    device=DEVICE
).build(
    complex_system,
    data=None,
    entity_subset=[0]
)

## 3. Score single mutants

We can now define the complex instance to be scored (in this particular case, we could also use `complex_system.rep_to_instance()` to achieve the same result as we set the WT sequences as `rep` on each of the entities):

In [9]:
target_inst = SystemInstance([
    EntityInstance(rep=vh_seq),
    EntityInstance(rep=vl_seq),
    EntityInstance(rep=antigen_seq)
])

We then perform single mutation scans with the `single_mutation_scan()` function. In this example, we will mutate the first entity of the complex instance, the heavy chain of the antibody (VH). To speed up the computation for this demo, we  just scan a subset of positions. By leaving the `positions` argument at the default value `None`, all positions in the entity will be scanned.

In [10]:
vh_scan_esm = esm.single_mutation_scan(
    target_inst, entity=0, positions=[2, 3, 4]
)

vh_scan_esm

Some weights of the model checkpoint at facebook/esm2_t33_650M_UR50D were not used when initializing EsmForMaskedLM: ['esm.embeddings.position_embeddings.weight']
- This IS expected if you are initializing EsmForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


A         C         D          E         F         G  \
entity pos ref                                                                
0      2   V   -4.500174 -7.034502 -6.284377  -5.823559 -6.237570 -5.963315   
       3   Q   -4.813040 -8.221459 -4.231486  -2.447863 -9.340475 -5.215446   
       4   L   -9.730852 -9.794907 -9.997735 -10.126611 -6.639863 -9.655082   

                       H         I          K         L         M          N  \
entity pos ref                                                                 
0      2   V   -8.287521 -1.082073  -7.001963 -3.804688 -5.555989  -6.948669   
       3   Q   -4.536812 -7.140356  -2.575920 -5.636361 -6.875429  -3.865236   
       4   L   -8.924232 -7.765018 -10.096931  0.000000 -6.541648 -10.474517   

                       P        Q         R         S         T         V  \
entity pos ref                                                              
0      2   V   -6.083308 -6.55799 -6.936087 -6.349372 -5.731109  0.000000   
       3   Q   -6.749483  0.00000 -3.955470 -4.811246 -4.762799 -4.922449   
       4   L   -6.248441 -7.72572 -7.727615 -7.232332 -9.531530 -6.420860   

                       W         Y  
entity pos ref                      
0      2   V   -8.760638 -8.320858  
       3   Q   -7.155417 -8.891275  
       4   L   -9.037384 -9.981421

In [11]:
vh_scan_mpnn = mpnn_model.single_mutation_scan(
    target_inst, entity=0, positions=[2, 3, 4]
)

vh_scan_mpnn


to                     A         C         D         E         F         G  \
entity pos ref                                                               
0      2   V   -0.000618 -0.002925 -0.002772 -0.001878 -0.002899 -0.002117   
       3   Q   -0.000966 -0.002714 -0.001859 -0.000343 -0.002309 -0.002166   
       4   L   -0.000987 -0.000917 -0.000230 -0.000187 -0.000650 -0.001247   

to                     H         I         K         L         M         N  \
entity pos ref                                                               
0      2   V   -0.002721 -0.000909 -0.002296 -0.002895 -0.000207 -0.002439   
       3   Q   -0.001104 -0.001823 -0.000115 -0.001925 -0.001892 -0.000500   
       4   L   -0.000428  0.001730 -0.001957  0.000000  0.000607 -0.000201   

to                     P         Q         R         S         T         V  \
entity pos ref                                                               
0      2   V   -0.000581 -0.001563 -0.002217 -0.001707 -0.001313  0.000000   
       3   Q   -0.002982  0.000000 -0.000353  0.000359  0.000409 -0.001030   
       4   L   -0.001422 -0.000462 -0.001278 -0.001026 -0.000095  0.002613   

to                     W         Y  
entity pos ref                      
0      2   V   -0.003297 -0.002776  
       3   Q   -0.002935 -0.002122  
       4   L   -0.000598 -0.000372

We could now perform downstream analysis by comparing the mutation effects from the two models.